# 🚀 FaceKey Animation Studio — Google Colab Cloud Data Collector
### Automated Silent YouTube Extraction + Google Drive (5 TB Plan) Chunking

**Features:**
- **5 TB Storage Optimized**: Seamlessly uploads `.tar.gz` chunks into your 5 TB Google Drive.
- **Parallel Multi-Worker Coordination**: Can run concurrently with Kaggle and Local PC workers without duplicate downloads!
- **Distributed Atomic Locks**: Locks URLs in Google Drive (`locks/<id>.lock.json`) with live heartbeats.
- **Auto-Reclaims Stale Locks**: If a session terminates unexpectedly or disconnects, abandoned locks are automatically recovered and resumed!
- **Zero Local Disk Bloat**: Local scratch memory is purged after each chunk to stay strictly under 5 GB.

## Step 1: Mount Google Drive (5 TB Plan)
Run this cell and click **Connect to Google Drive**.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/FaceKeyDataset'
os.makedirs(f'{DRIVE_DIR}/chunks', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/locks', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)
print(f'[+] Google Drive mounted successfully at: {DRIVE_DIR}')

## Step 2: Clone FaceKey Repository & Install Fast Cloud Dependencies

In [ ]:
# Clone or pull latest code from your GitHub repository
REPO_URL = 'https://github.com/Vijay-1010110/FaceKeyAnimation.git'
APP_DIR = '/content/FaceKeyAnimation'

if os.path.exists(APP_DIR):
    !cd {APP_DIR} && git pull
else:
    !git clone {REPO_URL} {APP_DIR}

%cd {APP_DIR}

# Install lightweight headless runtime dependencies
!pip install -q yt-dlp mediapipe opencv-python-headless numpy sounddevice

## Step 3: Inspect / Add YouTube URLs in Your Colab Queue
Colab uses its dedicated queue `youtube_urls_colab.txt` inside `MyDrive/FaceKeyDataset/`.
You can paste links here or directly edit the text file in Google Drive. Even while running, new links are detected dynamically!

In [ ]:
urls_file = f'{DRIVE_DIR}/youtube_urls_colab.txt'

if not os.path.exists(urls_file):
    # Copy from repo if not on drive yet
    import shutil
    repo_colab = os.path.join(APP_DIR, 'sessions', 'youtube_urls_colab.txt')
    if os.path.exists(repo_colab):
        shutil.copy2(repo_colab, urls_file)

print(f'[+] Colab Queue file ready on Google Drive: {urls_file}')
with open(urls_file, 'r', encoding='utf-8') as f:
    lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]
    print(f'[*] Active URLs in Colab Queue: {len(lines)} videos queued')
    print('First 5 links:')
    for u in lines[:5]:
        print('  -', u)

## Step 4: Run Silent 480p Cloud Batch Data Collector
This runs completely in the background on Google Colab:
- Identified as `colab-worker-1`.
- Claims locks in Drive before processing to prevent overlap with Kaggle.
- Heartbeat keeps lock active; auto-reclaims stale locks if interrupted.
- Packages data into compressed `.tar.gz` chunks in your 5 TB Google Drive.
- Cleans local Colab disk to maintain < 5 GB usage.

In [ ]:
!python scripts/cloud_data_collector.py \
    --drive-dir "/content/drive/MyDrive/FaceKeyDataset" \
    --worker-id "colab-worker-1" \
    --urls-file "/content/drive/MyDrive/FaceKeyDataset/youtube_urls_colab.txt" \
    --chunk-size 5 \
    --quality 480p